# Training YOLOv8n Deteksi Fertilitas Telur (Fertile vs Infertile)
Notebook Google Colab ini dirancang untuk mengunduh, menggabungkan dataset secara lokal di `/content`, dan melatih model **YOLOv8n** (Ultralytics) untuk mendeteksi 2 kelas:
- `0`: **fertile** (telur fertil/berembrio)
- `1`: **infertile** (telur infertil/tidak berkembang)

### Penyimpanan:
- **Dataset (Unduh & Merge)**: Disimpan di local runtime `/content/datasets` dan `/content/merged_egg_dataset` untuk performa I/O training maksimal dan bebas bottleneck.
- **Model & Checkpoint**: Disimpan di Google Drive (`/content/drive/MyDrive/egg_fertility_models`) agar bobot model tetap aman dan tersimpan permanen.

### Alur Kerja:
1. Cek ketersediaan GPU Google Colab (T4 / V100 / A100) & install library `ultralytics`.
2. Mount Google Drive (hanya untuk menyimpan model & checkpoint training).
3. Unduh dataset (Kaggle & Roboflow) langsung ke `/content/datasets`.
4. Deteksi folder dataset dan verifikasi konfigurasi kelas.
5. Harmonisasi Label & Penggabungan data split (`train`, `valid`, `test`) ke `/content/merged_egg_dataset`.
6. Ringkasan & verifikasi dataset gabungan (`data.yaml`).
7. Training YOLOv8n (`yolov8n.pt`) dengan checkpoint otomatis tersimpan ke Google Drive.
8. Evaluasi metrik performa (`mAP@50`, `mAP@50-95`, Precision, Recall).
9. Visualisasi grafik metrik pelatihan & confusion matrix.
10. Inferensi prediksi pada sampel data uji (`test`).
11. Export model ke format ONNX dan backup bobot terbaik ke Google Drive.

## 1. Verifikasi Akselerator GPU & Instalasi Dependensi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cek ketersediaan GPU di Google Colab
import torch
print("=" * 45)
if torch.cuda.is_available():
    print(f"GPU Aktif   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("PERINGATAN: GPU tidak terdeteksi.")
    print("    Aktifkan GPU di: Runtime -> Change runtime type -> T4 GPU")
print("=" * 45)

# Install Ultralytics YOLOv8 dan library pendukung
!pip install -q ultralytics pyyaml opencv-python matplotlib tqdm pillow roboflow python-dotenv

## 2. Mount Google Drive (Khusus Penyimpanan Model)
Google Drive **hanya digunakan untuk menyimpan checkpoint model** (`best.pt`, `last.pt`), log runs, serta export model ONNX agar tidak hilang saat runtime Colab berakhir/terputus.
Dataset mentah dan hasil merge disimpan di `/content` lokal agar proses training berjalan cepat tanpa kendala I/O timeout.

In [ ]:
# Install library yang dibutuhkan
!pip install -q roboflow kagglehub pyyaml

# Connect ke Google Drive (hanya untuk menyimpan model)
import os
from pathlib import Path
from google.colab import drive

USE_DRIVE = True
DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/egg_fertility_models")

if USE_DRIVE:
    try:
        drive.mount('/content/drive')
        DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive berhasil terhubung. Model akan disimpan di: {DRIVE_MODEL_DIR}")
    except Exception as e:
        print(f"Gagal mount Drive: {e}")

## 3. Dowload Dataset

In [ ]:
import os
import shutil
from pathlib import Path
import kagglehub
from roboflow import Roboflow

class DatasetManager:
    def __init__(self, base_save_dir="/content/datasets"):
        self.base_dir = Path(base_save_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        os.environ['KAGGLEHUB_CACHE'] = str(self.base_dir / ".cache_kaggle")

    def _download_kaggle(self, config, target_dir):
        print(f"Mengunduh [{config['name']}] dari Kaggle...")
        raw_path = kagglehub.dataset_download(config['source_id'])
        raw_path = Path(raw_path)
        if not target_dir.exists():
            shutil.copytree(raw_path, target_dir)
        print(f"Selesai! Tersimpan di: {target_dir}")

    def _download_roboflow(self, config, target_dir):
        print(f"Mengunduh [{config['name']}] dari Roboflow...")

        original_cwd = os.getcwd()
        os.chdir(self.base_dir)

        try:
            rf = Roboflow(api_key=config['api_key'])
            project = rf.workspace(config['workspace']).project(config['project'])
            version = project.version(config['version'])

            dataset = version.download(config['format'])

            downloaded_folder = Path(dataset.location)

            if downloaded_folder.exists() and downloaded_folder != target_dir:
                if target_dir.exists():
                    shutil.rmtree(target_dir)
                shutil.move(str(downloaded_folder), str(target_dir))

            print(f"Selesai! Tersimpan di: {target_dir}")

        except Exception as e:
            print(f"Error saat download Roboflow: {e}")
            raise
        finally:
            os.chdir(original_cwd)

    def download_dataset(self, config):
        target_dir = self.base_dir / config['name']

        if target_dir.exists() and any(target_dir.iterdir()):
            print(f"Dataset [{config['name']}] sudah ada di {target_dir}. Melewati unduhan.")
            return target_dir

        source_type = config['source_type'].lower()
        if source_type == 'kaggle':
            self._download_kaggle(config, target_dir)
        elif source_type == 'roboflow':
            self._download_roboflow(config, target_dir)
        else:
            raise ValueError(f"Source type '{source_type}' tidak didukung.")

        return target_dir

    def download_all_datasets(self, datasets_config):
        results = {}
        for config in datasets_config:
            try:
                path = self.download_dataset(config)
                results[config['name']] = {
                    'status': 'success',
                    'path': str(path)
                }
            except Exception as e:
                results[config['name']] = {
                    'status': 'failed',
                    'error': str(e)
                }
                print(f"Gagal download {config['name']}: {e}")
        return results

In [ ]:
# Inisialisasi API Key Roboflow secara aman
import os
import getpass
import subprocess
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --- Install roboflow jika belum ada ---
try:
    import roboflow  # noqa
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])

from roboflow import Roboflow

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass.getpass("Masukkan Roboflow API Key: ")

# Inisialisasi manager dengan folder simpan lokal di /content
DATASET_BASE_DIR = Path("/content/datasets")
manager = DatasetManager(base_save_dir=DATASET_BASE_DIR)

# Daftar konfigurasi dataset
DATASETS_CONFIG = [
    {
        "name": "duck-egg-detection-roboflow",
        "source_type": "roboflow",
        "api_key": ROBOFLOW_API_KEY,
        "workspace": "samsuri-2ug2l",
        "project": "duck-egg-detection",
        "version": 2,
        "format": "yolov8"
    },
    {
        "name": "egg-fertility-candling-roboflow",
        "source_type": "roboflow",
        "api_key": ROBOFLOW_API_KEY,
        "workspace": "nodkeyobma",
        "project": "egg-fertility-candling",
        "version": 9,
        "format": "yolov8"
    },
    {
        "name": "egg-fertility-nitkl",
        "source_type": "roboflow",
        "api_key": ROBOFLOW_API_KEY,
        "workspace": "electronic-engineering-polytechnic-institute-of-surabaya-zxfjf",
        "project": "egg-fertility-nitkl",
        "version": 9,
        "format": "yolov8"
    }
]

# Download semua dataset ke /content/datasets
print("Memulai download semua dataset ke /content/datasets...")
print("=" * 50)
results = manager.download_all_datasets(DATASETS_CONFIG)

# Tampilkan hasil
print("\n" + "=" * 50)
print("HASIL DOWNLOAD:")
print("=" * 50)
for name, result in results.items():
    if result['status'] == 'success':
        print(f"SUKSES - {name}: {result['path']}")
    else:
        print(f"GAGAL - {name}: {result['error']}")

print("\n" + "=" * 50)
print(f"Total dataset: {len(results)}")
success_count = sum(1 for r in results.values() if r['status'] == 'success')
print(f"Berhasil: {success_count}/{len(results)}")
print("=" * 50)

Deteksi Folder Dataset & Pengecekan Kelas


In [ ]:
import yaml
from pathlib import Path

# Base directory dataset di /content
base_dir = DATASET_BASE_DIR if 'DATASET_BASE_DIR' in globals() else Path("/content/datasets")

print("Verifikasi Dataset & Konfigurasi Kelas:")
for config in DATASETS_CONFIG:
    ds_path = base_dir / config['name']
    print(f"\n Dataset: {config['name']}")
    print(f"   Path: {ds_path}")

    # Cari file data.yaml
    yaml_files = list(ds_path.glob("*.yaml")) + list(ds_path.glob("*.yml"))
    if yaml_files:
        with open(yaml_files[0], 'r') as f:
            data = yaml.safe_load(f)

        # Format nama kelas
        names = data.get('names', [])
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names.keys())]

        print(f"   File Config : {yaml_files[0].name}")
        print(f"   Daftar Kelas: {names}")
    else:
        print("   File data.yaml TIDAK DITEMUKAN!")

## 4. Penggabungan Dataset & Penyelarasan ID Kelas (Remapping)
Standar kelas final yang akan dipelajari oleh model:
- Kelas `0` : **`fertile`**
- Kelas `1` : **`infertile`**

Skrip ini akan:
1. Menganalisis urutan label pada masing-masing dataset dan memetakannya ke standar `0: fertile` dan `1: infertile`.
2. Menyalin gambar dan label secara lokal ke `/content/merged_egg_dataset`.
3. Menambahkan prefix `ds1_`, `ds2_`, `ds3_` pada nama file agar tidak ada gambar yang tertimpa (*name collision*).
4. Menulis ulang file `.txt` label dengan ID kelas yang sudah diselaraskan.

In [ ]:
import os
import shutil
import re
import yaml
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Simpan dan gabungkan dataset di /content (lokal)
DATASET_BASE_DIR = Path("/content/datasets")
MERGED_DIR = Path("/content/merged_egg_dataset")
TARGET_CLASSES = ["fertile", "infertile"]

for split in ['train', 'valid', 'test']:
    (MERGED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

def build_remap_dict(ds_path):
    yaml_files = list(ds_path.glob("*.yaml")) + list(ds_path.glob("*.yml"))
    classes = []

    if yaml_files:
        with open(yaml_files[0], 'r') as f:
            data = yaml.safe_load(f)
            names = data.get('names', [])
            if isinstance(names, dict):
                classes = [names[k] for k in sorted(names.keys())]
            else:
                classes = names

    remap = {}
    for old_id, name in enumerate(classes):
        name_clean = str(name).strip().lower()

        if re.search(r'\b(inf|infertile|in-fertile|unfertile|unfertilized|tidak-fertil)\b', name_clean):
            remap[old_id] = 1
        elif re.search(r'\b(fer|fertile|fertilized|fertil)\b', name_clean):
            remap[old_id] = 0
        else:
            remap[old_id] = 1 if 'inf' in name_clean else 0

    print(f"Pemetaan [{ds_path.name}]: {classes} -> {remap}")
    return remap

def merge_datasets(datasets_config):
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    overall_stats = defaultdict(int)

    for idx, config in enumerate(datasets_config):
        ds_path = DATASET_BASE_DIR / config['name']
        if not ds_path.exists():
            print(f"Warning: Folder {ds_path} tidak ditemukan, melewati...")
            continue

        prefix = f"ds{idx+1}"
        remap_dict = build_remap_dict(ds_path)

        for split in ['train', 'valid', 'test']:
            src_img_dir = ds_path / split / 'images'
            src_lbl_dir = ds_path / split / 'labels'

            if not src_img_dir.exists() and (ds_path / split).exists():
                src_img_dir = ds_path / split
                src_lbl_dir = ds_path / split

            if not src_img_dir.exists():
                continue

            dst_img_dir = MERGED_DIR / split / 'images'
            dst_lbl_dir = MERGED_DIR / split / 'labels'

            images = [f for f in src_img_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_exts]

            for img_p in tqdm(images, desc=f"Merging {config['name']} [{split}]"):
                new_stem = f"{prefix}_{img_p.stem}"
                new_img_name = f"{new_stem}{img_p.suffix.lower()}"

                shutil.copy2(img_p, dst_img_dir / new_img_name)
                overall_stats[f"{split}_images"] += 1

                src_lbl_p = src_lbl_dir / f"{img_p.stem}.txt"
                dst_lbl_p = dst_lbl_dir / f"{new_stem}.txt"

                if src_lbl_p.exists():
                    new_lines = []
                    with open(src_lbl_p, 'r') as f_in:
                        for line in f_in:
                            parts = line.strip().split()
                            if not parts:
                                continue
                            try:
                                old_cls = int(float(parts[0]))
                                new_cls = remap_dict.get(old_cls, 0)
                                new_lines.append(f"{new_cls} " + " ".join(parts[1:]))
                            except ValueError:
                                continue

                    with open(dst_lbl_p, 'w') as f_out:
                        for nl in new_lines:
                            f_out.write(nl + "\n")
                else:
                    dst_lbl_p.touch()

    merged_yaml = {
        'path': str(MERGED_DIR.resolve()),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(TARGET_CLASSES),
        'names': TARGET_CLASSES
    }

    yaml_path = MERGED_DIR / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(merged_yaml, f, sort_keys=False)

    print(f"\nPenggabungan selesai.")
    print(f"Dataset gabungan disimpan di: {MERGED_DIR}")
    print(f"File config: {yaml_path}")

merge_datasets(DATASETS_CONFIG)

## 5. Ringkasan & Verifikasi Dataset Gabungan
Mengecek jumlah gambar dan distribusi kelas pada tiap data split (`train`, `valid`, `test`).


In [ ]:
from pathlib import Path

yaml_path = MERGED_DIR / 'data.yaml'

print("=" * 55)
print("             STATISTIK DATASET GABUNGAN")
print("=" * 55)

for split in ['train', 'valid', 'test']:
    img_list = list((MERGED_DIR / split / 'images').glob('*.*'))
    lbl_list = list((MERGED_DIR / split / 'labels').glob('*.txt'))

    cls_counts = {0: 0, 1: 0}
    for lbl in lbl_list:
        if lbl.exists():
            with open(lbl, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        try:
                            c = int(parts[0])
                            cls_counts[c] = cls_counts.get(c, 0) + 1
                        except ValueError:
                            pass

    print(f"Split {split.upper():<6} -> {len(img_list):>5} Gambar | Fertile: {cls_counts.get(0, 0):>4} | Infertile: {cls_counts.get(1, 0):>4}")

print("=" * 55)
print(f"\nKonten {yaml_path}:")
with open(yaml_path, 'r') as f:
    print(f.read())

## 6. Training Model YOLOv8n
- **Model**: `yolov8n.pt` (YOLOv8 Nano pretrained - ringan, cepat, dan akurat).
- **Input Size**: `640` piksel.


In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

# Set device
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"Training berjalan pada Device: {DEVICE}")

# Inisialisasi model pre-trained YOLOv8
model = YOLO('yolov8n.pt')

# Parameter Pelatihan yang Dioptimasi
EPOCHS = 60  # Cukup untuk yolov8n dengan dataset 6k+ gambar
IMG_SIZE = 640
BATCH_SIZE = 48  # Sesuaikan dengan VRAM, turunkan jika OOM

# Simpan hasil training
DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/egg_fertility_models")
PROJECT_DIR = str(DRIVE_MODEL_DIR / 'runs')
EXPERIMENT_NAME = 'yolov8n_fertile_infertile_optimized'

# Eksekusi Pelatihan
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT_DIR,
    name=EXPERIMENT_NAME,
    save=True,
    plots=True,
    patience=20,  # Stop lebih awal jika tidak ada improvement (default=100)
    workers=4,
    cache="disk",

    # OPTIMASI UTAMA - Gunakan default terbaik YOLOv8
    optimizer='auto',  # YOLO pilih otomatis (SGD atau AdamW)
    lr0=0.01,  # Learning rate awal (default YOLO)
    lrf=0.01,  # Final learning rate factor (default)
    momentum=0.937,  # SGD momentum (default)
    weight_decay=0.0005,  # Default YOLO

    # COS LR SCHEDULER (rekomendasi untuk konvergensi lebih baik)
    cos_lr=True,  # Aktifkan cosine annealing

    # WARMUP (penting untuk stabilitas training)
    warmup_epochs=3,  # Default YOLO
    warmup_momentum=0.8,  # Default
    warmup_bias_lr=0.1,  # Default

    # AUGMENTASI DASAR (cukup untuk dataset Anda)
    # hsv_h=0.015,  # Hue (default)
    # hsv_s=0.7,    # Saturation (default)
    # hsv_v=0.4,    # Value (default)
    # fliplr=0.5,   # Horizontal flip (default)
    # mosaic=1.0,   # Mosaic augmentasi (default)

    # NONAKTIFKAN augmentasi yang tidak perlu untuk mempercepat
    # degrees=0.0,
    # translate=0.0,
    # scale=0.0,
    # shear=0.0,
    # perspective=0.0,
    # flipud=0.0,  # Vertical flip tidak berguna untuk telur

    # MOSAIC CLOSE (untuk stabilitas di akhir training)
    close_mosaic=15,  # Matikan mosaic 10 epoch terakhir

    # LABEL SMOOTHING (mencegah overfitting)

    # MIXUP (sedikit membantu generalisasi)
    mixup=0.0,  # Nonaktifkan dulu, aktifkan jika perlu

    # DROPOUT (untuk mencegah overfitting)
    dropout=0.0,  # Default untuk yolov8n

    # NMS & POST-PROCESSING
    nbs=64,  # Nominal batch size (default)
    overlap_mask=True,  # Untuk segmentasi (tidak digunakan)
    mask_ratio=4,  # Untuk segmentasi

    # VERBOSE
    verbose=True,

    # FRAGMENT (opsional untuk mempercepat)
    # rect=False,  # Training rectangular (lebih cepat)
    # resume=False,
    amp=True,  # Mixed precision (lebih cepat)
)

print(f"\nPelatihan selesai.")
print(f"Hasil dan checkpoint model tersimpan di Drive: {results.save_dir}")

## 7. Evaluasi Model pada Data Validasi (Validation Set)
Mengekstrak metrik evaluasi deteksi objek standar:
- **mAP@50**: Mean Average Precision pada IoU threshold 0.50
- **mAP@50-95**: Rata-rata mAP pada IoU threshold 0.50 hingga 0.95
- **Precision & Recall** untuk kedua kelas (`fertile` dan `infertile`)


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# 1. Cari folder eksperimen yang SUDAH memiliki file best.pt (folder kosong/-4 akan diabaikan)
valid_dirs = [
    d for d in Path(PROJECT_DIR).iterdir()
    if d.is_dir() and d.name.startswith(EXPERIMENT_NAME) and (d / 'weights' / 'best.pt').exists()
]

if valid_dirs:
    # Pilih folder dengan file best.pt paling baru (otomatis memilih folder -3)
    valid_dirs.sort(key=lambda p: (p / 'weights' / 'best.pt').stat().st_mtime, reverse=True)
    actual_exp_dir = valid_dirs[0]
else:
    # Jika tidak ada, coba langsung ke folder -3
    actual_exp_dir = Path(PROJECT_DIR) / f"{EXPERIMENT_NAME}-3"

actual_exp_name = actual_exp_dir.name
best_model_path = actual_exp_dir / 'weights' / 'best.pt'

print(f"Eksperimen aktif : {actual_exp_name}")
print(f"Memuat model dari: {best_model_path}")
best_model = YOLO(str(best_model_path))

# 2. Jalankan evaluasi pada data validasi
val_metrics = best_model.val(data=str(yaml_path), split='val')

print("\n" + "=" * 45)
print("             METRIK EVALUASI MODEL")
print("=" * 45)
print(f"Precision          : {val_metrics.box.mp:.4f}")
print(f"Recall             : {val_metrics.box.mr:.4f}")
print(f"mAP @ 0.50         : {val_metrics.box.map50:.4f}")
print(f"mAP @ 0.50:0.95    : {val_metrics.box.map:.4f}")
print("-" * 45)

for idx, cls_name in enumerate(TARGET_CLASSES):
    if idx < len(val_metrics.box.ap50):
        map50_cls = val_metrics.box.ap50[idx]
        map_all_cls = val_metrics.box.maps[idx]
        print(f"[{cls_name:<10}] mAP@50: {map50_cls:.4f} | mAP@50-95: {map_all_cls:.4f}")

print("=" * 45)

## 8. Visualisasi Grafik Performa & Confusion Matrix
Menampilkan kurva loss, precision-recall curve, dan confusion matrix hasil training YOLOv8.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Gunakan direktori eksperimen aktif yang ditemukan di evaluasi
exp_folder = actual_exp_dir if 'actual_exp_dir' in globals() else Path(PROJECT_DIR) / actual_exp_name

chart_files = [
    ("Grafik Metrik Training & Loss", exp_folder / "results.png"),
    ("Confusion Matrix (Normalized)", exp_folder / "confusion_matrix_normalized.png"),
    ("Kurva F1-Confidence", exp_folder / "F1_curve.png"),
    ("Kurva Precision-Recall", exp_folder / "PR_curve.png")
]

for title, file_path in chart_files:
    if file_path.exists():
        img = Image.open(file_path)
        figsize = (12, 8) if "results" in file_path.name else (8, 6)
        plt.figure(figsize=figsize)
        plt.imshow(img)
        plt.title(title, fontsize=12, fontweight='bold', pad=10)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"File {file_path.name} tidak ditemukan di {exp_folder}.")

## 9. Uji Inferensi Prediksi pada Sampel Data Uji (`test`)
Menguji model terbaik (`best.pt`) pada gambar dari split `test` untuk melihat visualisasi bounding box, label kelas (`fertile`/`infertile`), dan confidence score.


In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# 1. Cari folder eksperimen terbaru secara otomatis
original_base_name = EXPERIMENT_NAME
exp_dirs = [d for d in Path(PROJECT_DIR).iterdir() if d.is_dir() and d.name.startswith(original_base_name)]

if exp_dirs:
    exp_dirs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    actual_exp_name = exp_dirs[0].name
    print(f"Menggunakan direktori eksperimen: {actual_exp_name}")
else:
    actual_exp_name = original_base_name
    print(f"Peringatan: Direktori eksperimen spesifik tidak ditemukan. Menggunakan nama dasar: {actual_exp_name}")

# 2. Muat model terbaik
best_model_path = Path(PROJECT_DIR) / actual_exp_name / 'weights' / 'best.pt'
infer_model = YOLO(str(best_model_path))

# 3. Ambil sampel gambar dari folder test (atau validasi sebagai cadangan)
test_dir = MERGED_DIR / 'test' / 'images'
test_imgs = list(test_dir.glob('*.*'))

if not test_imgs:
    test_dir = MERGED_DIR / 'valid' / 'images'
    test_imgs = list(test_dir.glob('*.*'))

# 4. Pilih 6 sampel acak
num_samples = min(6, len(test_imgs))
sample_images = random.sample(test_imgs, num_samples)

# 5. Visualisasi Hasil Prediksi
plt.figure(figsize=(16, 10))
for i, img_path in enumerate(sample_images):
    # Prediksi dengan threshold confidence 0.4
    results = infer_model.predict(source=str(img_path), conf=0.4, iou=0.5)[0]

    # Render bounding box bawaan YOLO (format BGR ke RGB via slicing)
    res_bgr = results.plot()
    res_rgb = res_bgr[..., ::-1]

    plt.subplot(2, 3, i + 1)
    plt.imshow(res_rgb)
    det_count = len(results.boxes)
    plt.title(f"{img_path.name}\nJumlah Deteksi: {det_count}", fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

## 10. Export Model & Unduh Bobot Terbaik (`best.pt`)
1. Export model ke format **ONNX** (sangat berguna untuk deployment di web, desktop, atau edge device seperti Raspberry Pi / Jetson).
2. Simpan cadangan model (`best.pt` dan `best.onnx`) ke Google Drive di folder model.
3. Unduh langsung file `best.pt` ke komputer lokal.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
from ultralytics import YOLO

# 1. Tentukan lokasi best.pt menggunakan folder eksperimen terbaru di Drive
best_pt = Path(PROJECT_DIR) / actual_exp_name / 'weights' / 'best.pt'
export_model = YOLO(str(best_pt))

# 2. Export ke format ONNX
print("Mengekspor model ke format ONNX...")
onnx_output_path = export_model.export(format='onnx', dynamic=False)
onnx_pt = Path(onnx_output_path)
print(f"Model ONNX tersimpan di: {onnx_pt}")

# 3. Backup model ke Google Drive
backup_dir = DRIVE_MODEL_DIR / 'models_exported'
backup_dir.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    shutil.copy2(best_pt, backup_dir / 'best.pt')
if onnx_pt.exists():
    shutil.copy2(onnx_pt, backup_dir / 'best.onnx')

print(f"Model berhasil di-backup ke Drive di: {backup_dir}")

# 4. Unduh otomatis ke lokal komputer (opsional)
print("Mengunduh file best.pt ke komputer lokal...")
files.download(str(best_pt))